# Why use *pyeodh* over *pystac*?
<style>
.scibtn {
  background-color: #04AA6D;
  color: white;
  font-size: 15px;
  padding: 12px;
  border: none;
  border-radius: 5px;
  cursor: pointer;
}
.techbtn {
  background-color: #0407aaff;
  color: white;
  font-size: 15px;
  padding: 12px;
  border: none;
  border-radius: 5px;
  cursor: pointer;
}
.databtn {
  background-color: #edfd03ff;
  color: black;
  font-size: 15px;
  padding: 12px;
  border: none;
  border-radius: 5px;
  cursor: pointer;
}

</style>

<button class="techbtn">Technical</button> 

__Description & purpose__: This notebook introduces how *pyeodh* can simplify Python acccess to our EODH STAC catalogue. If you are normally a *pystac* user, this guide should help to familiarise you with our custom Python API client, *pyeodh*.

## Set-up

In [ ]:
%pip install pystac_client # install required package

In [ ]:
# import both required packages
from pystac_client import Client
import pyeodh

# pystac

### Understand the EODH STAC catalogue structure

<style>
.howbtn {
  background-color: #f7871fff;
  color: white;
  font-size: 15px;
  padding: 12px;
  border: none;
  border-radius: 5px;
  cursor: pointer;
}
</style>

<button class="howbtn">How-to Highlight</button>

Start by accessing the main EODH catalog root.

In [5]:
cat_tac_api_root_endpoint = "https://eodatahub.org.uk/api/catalogue/stac/" #root starting point
main_cat = Client.open(url=cat_tac_api_root_endpoint) # calls the selected url
main_cat

<Client id=stac-fastapi>

📂 Within the root catalog, search for sub catalogs at this level.

In [6]:
# Check for Sub-Catalogs first
print("Searching for Sub-Catalogs...")
for link in main_cat.get_links():
    if link.rel == 'child':
        print(f"Found Sub-Catalog: {link.href}")


Searching for Sub-Catalogs...
Found Sub-Catalog: https://eodatahub.org.uk/api/catalogue/stac/catalogs/commercial
Found Sub-Catalog: https://eodatahub.org.uk/api/catalogue/stac/catalogs/public
Found Sub-Catalog: https://eodatahub.org.uk/api/catalogue/stac/catalogs/supported-datasets
Found Sub-Catalog: https://eodatahub.org.uk/api/catalogue/stac/catalogs/user


We find a number of sub catalogs nested within the root catalog. To help us understand where our data collection can be found, lets first print out a visual diagram of the directory structure for the EODH STAC catalogue.

In [7]:
# We see whats inside and get the STAC metadata collection IDs
def find_collections(catalog, depth=0):
    indent = "  " * depth
    # Try to get collections in this specific catalog
    try:
        cols = list(catalog.get_collections())
        for c in cols:
            print(f"{indent}✅ Collection Found: {c.id}")
    except Exception:
        pass

    # Look into children (Sub-Catalogs)
    for child_link in catalog.get_children():
        print(f"{indent}📂 Entering Sub-Catalog: {child_link.id}")
        find_collections(child_link, depth + 1)

find_collections(main_cat)

✅ Collection Found: ukcp
✅ Collection Found: sentinel2_ard
✅ Collection Found: sentinel1_ard
✅ Collection Found: sentinel1
✅ Collection Found: qa_radiometric
✅ Collection Found: qa_radiometric
✅ Collection Found: qa_radiometric
✅ Collection Found: qa_documentation
✅ Collection Found: qa_documentation
✅ Collection Found: qa_documentation
✅ Collection Found: land_cover
✅ Collection Found: era5_repack
✅ Collection Found: eocis-sst-cdrv3
✅ Collection Found: eocis-soil-moisture-africa
✅ Collection Found: eocis-lst-s3b-night
✅ Collection Found: eocis-lst-s3b-day
✅ Collection Found: eocis-lst-s3a-night
✅ Collection Found: eocis-lst-s3a-day
✅ Collection Found: eocis-chuk-land-vegetation-lai
✅ Collection Found: eocis-chuk-land-vegetation-fapar
✅ Collection Found: eocis-chuk-geospatial-landcover
✅ Collection Found: eocis-chuk-geospatial-landclass
✅ Collection Found: eocis-chuk-geospatial-elevation
✅ Collection Found: eocis-chuk-geospatial-builtarea
✅ Collection Found: eocis-chuk-geospatial
✅ Col

### Access a data collection with *pystac*

<style>
.howbtn {
  background-color: #f7871fff;
  color: white;
  font-size: 15px;
  padding: 12px;
  border: none;
  border-radius: 5px;
  cursor: pointer;
}
</style>

<button class="howbtn">How-to Highlight</button>

To find the ```sentinel2_ard``` data collection, we should look inside the sub catalogs. Sentinel 2 ARD can be found within the sub catalog called ```public```, according to our directory structure.

In [13]:
cat_pub_root_endpoint = "https://eodatahub.org.uk/api/catalogue/stac/catalogs/public/" #root starting point
public_cat = Client.open(url=cat_pub_root_endpoint) # calls the selected url
public_cat

<Client id=public>

Similar to entering a folder, we use ```get_child``` to look inside that ```public``` catalog.

In [14]:
sub2 = main_cat.get_child('public')
sub2

<Client id=public>

🔎 Going one level deeper into the directory structure, we find...

In [10]:
sub3 = sub2.get_child('ceda-stac-catalogue')
sub3

<Client id=ceda-stac-catalogue>

We try looking for our collection here, inside the ```ceda-stac-catalogue```

In [11]:
sentinel_coll = sub3.get_collection('sentinel2_ard')
sentinel_coll

<CollectionClient id=sentinel2_ard>

We have found the ```sentinel2_ard``` collection! We can also reach the data collection using the ```get_child``` arguement too.

In [12]:
sub4 = sub3.get_child('sentinel2_ard')
sub4

<CollectionClient id=sentinel2_ard>

<div class="alert alert-block alert-warning"> 
<b>💭 Reflections </b>
<br>
- Using pystac to access the EODH STAC catalogue was quite difficult, and required many lines of code! <br>
- Now, let's understand why using pyeodh is a better choice that makes accessing the data collection much simpler <br>

</div>

# pyeodh

🔌 First, we reach the data Hub by connecting via the Python client.

In [18]:
client = pyeodh.Client(
    base_url="https://eodatahub.org.uk"
).get_catalog_service()

### Access a data collection with *pyeodh*
Using inbuild functions of the pyeodh package, we can quickly fetch the catalog and the collection we are looking for. Based on our printed directory structure, we already know the filepath.

In [22]:
s2ard = client.get_catalog("public/catalogs/ceda-stac-catalogue").get_collection('sentinel2_ard')
s2ard

<style>
.howbtn {
  background-color: #f7871fff;
  color: white;
  font-size: 15px;
  padding: 12px;
  border: none;
  border-radius: 5px;
  cursor: pointer;
}
</style>

<button class="howbtn">How-to Highlight</button>

Immediately we can print some items, so we know that we have successfully found our data collection!

In [23]:
items = s2ard.get_items()

# Warning: without the limit to 10 items this will take a long time for large catalogues such as s2ard
for item in items[:10]:
    print(item.id)

neodc.sentinel_ard.data.sentinel_2.2026.02.12.S2A_20260212_latn608lonw0002_T30VXN_ORB037_20260212145709_utm30n_osgb
neodc.sentinel_ard.data.sentinel_2.2026.02.12.S2B_20260212_latn608lonw0002_T30VXN_ORB037_20260212145332_utm30n_osgb
neodc.sentinel_ard.data.sentinel_2.2026.02.12.S2B_20260212_latn599lonw0020_T30VWM_ORB037_20260212145332_utm30n_osgb
neodc.sentinel_ard.data.sentinel_2.2026.02.12.S2B_20260212_latn599lonw0002_T30VXM_ORB037_20260212145332_utm30n_osgb
neodc.sentinel_ard.data.sentinel_2.2026.02.12.S2B_20260212_latn590lonw0020_T30VWL_ORB037_20260212145332_utm30n_osgb
neodc.sentinel_ard.data.sentinel_2.2026.02.12.S2B_20260212_latn572lonw0037_T30VVJ_ORB037_20260212145332_utm30n_osgb
neodc.sentinel_ard.data.sentinel_2.2026.02.12.S2B_20260212_latn510lonw0036_T30UVB_ORB037_20260212145332_utm30n_osgb
neodc.sentinel_ard.data.sentinel_2.2026.02.11.S2C_20260211_latn509lonw0008_T30UXB_ORB094_20260211125907_utm30n_osgb
neodc.sentinel_ard.data.sentinel_2.2026.02.11.S2C_20260211_latn509lone00

---

__Author(s)__: Gemma Newbold, Gisela Romero Candanedo, Alastair Graham

__Date created__: 2026-02-18

__Date last modified__: 2026-03-16

__Licence__: This notebook is licensed under Creative Commons Attribution-ShareAlike 4.0 International. The code is released using the BSD-2-Clause license.

<span style="font-size:0.65em;">
Copyright &copy; - All rights reserved.

Redistribution and use in source and binary forms, with or without modification, are permitted provided that the following conditions are met:

Redistributions of source code must retain the above copyright notice, this list of conditions and the following disclaimer. Redistributions in binary form must reproduce the above copyright notice, this list of conditions and the following disclaimer in the documentation and/or other materials provided with the distribution. THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS “AS IS” AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY, OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.</span>
